# Agenda

1. More with indexes
2. dtypes
3. `NaN`

# Indexes

An index is how we can access/set elements of our series. By default, the index is a "range index," meaning that it starts at 0 and goes up to the length - 1. This is similar to what we've seen in strings, lists, and tuples.

But we can also set the index to be anything we want. And that includes *anything*. This allows us to retrieve one or more elements from series that match our index.

In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

In [2]:
s = Series([10, 20, 30, 40, 50])
s

0    10
1    20
2    30
3    40
4    50
dtype: int64

In [3]:
s.index   # show me the index for this series

RangeIndex(start=0, stop=5, step=1)

In [4]:
s = Series([10, 20, 30, 40, 50],
          index=list('abcde'))
s

a    10
b    20
c    30
d    40
e    50
dtype: int64

In [5]:
s = Series([10, 20, 30, 40, 50],
          index='this is a sample index'.split())
s

this      10
is        20
a         30
sample    40
index     50
dtype: int64

In [6]:
s.loc['this']

np.int64(10)

In [7]:
s.loc['sample']

np.int64(40)

In [8]:
# fancy indexing
s.loc[['this', 'sample']]

this      10
sample    40
dtype: int64

In [9]:
s.iloc[0]

np.int64(10)

In [10]:
s.iloc[3]

np.int64(40)

In [11]:
s.iloc[[0, 3]]

this      10
sample    40
dtype: int64

# How fast does it run?

Python comes with a module called `timeit` that lets you time how long some code runs. I normally don't run it directly, but rather use Jupyter's "magic commands" -- they all start with `%` and let you do special things in your cell.

For example, if you say `%timeit`, then the rest of the line is run as code a lot of times, and it tells you the mean ± std.

If you say `%%timeit` at the start of a cell in a cell by itself, then you will do the same thing, but for all of the code in the cell.

In [12]:
%timeit s.loc[['this', 'sample']]

116 μs ± 1.75 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [13]:
%timeit s.iloc[[0, 3]]

25.7 μs ± 390 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [14]:
# we can also set values using `.loc` and `.iloc`

s.loc['sample'] = 999
s

this       10
is         20
a          30
sample    999
index      50
dtype: int64

In [15]:
s.iloc[[0, 4]] = 888   # I can assign with fancy indexing!

In [16]:
s

this      888
is         20
a          30
sample    999
index     888
dtype: int64

In [18]:
# boolean indexing

s.loc[ s > 100 ]    # this boolean index keeps only the values that are True

this      888
sample    999
index     888
dtype: int64

In [20]:
# we can set via the boolean/mask index

# this means:
# 1. find the elements that are > 100, get a boolean series back
# 2. Use that boolean series as a mask index in s.loc to find those elements
# 3. Assign to those elements, replacing the previous values

s.loc[ s > 100 ] = 777
s

this      777
is         20
a          30
sample    777
index     777
dtype: int64

In [24]:
# change all large numbers into the s.min()

s.loc[ s > 100 ] = s.min()
s

this      20
is        20
a         30
sample    20
index     20
dtype: int64

# Setting the index after the fact

If you have a series and want to set its index to something new, you can! Just assign to `s.index`. If it's an iterable (e.g., list or series), then it'll just be applied as the index.

Note that it must be of the right length!

In [25]:
s

this      20
is        20
a         30
sample    20
index     20
dtype: int64

In [26]:
s.index = list('vwxyz')
s

v    20
w    20
x    30
y    20
z    20
dtype: int64

In [27]:
s.index = Series([12, 34, 56, 78, 90])
s

12    20
34    20
56    30
78    20
90    20
dtype: int64

In [28]:
# can indexes contain repeated values?
# answer: yes!

s = Series([10, 20, 30, 40, 50, 60],
           index=list('abcdab'))
s

a    10
b    20
c    30
d    40
a    50
b    60
dtype: int64

In [29]:
s.loc['c']  # only one value has this index

np.int64(30)

In [30]:
s.loc['a']  # two values have this index

a    10
a    50
dtype: int64

# Exercise: Indexes

1. Create a Pandas series of 10 items, integers representing the high temps from the coming 10 days. The values should be ints, and the index should be strings -- the three-letter day name for that day.
2. What is the mean temperature on Wednesdays?
3. What is the mean temperature on Thursdays and Tuesdays?
4. What is the max temp in the first 3 days?

https://practice.lernerpython.com/classroom/ce06963a8d/ex-51

In [31]:
s = Series([34, 35, 35, 31, 30, 31, 32, 31, 31, 31],
           index='Tue Wed Thu Fri Sat Sun Mon Tue Wed Thu'.split())
s

Tue    34
Wed    35
Thu    35
Fri    31
Sat    30
Sun    31
Mon    32
Tue    31
Wed    31
Thu    31
dtype: int64

In [36]:
s.loc['Wed'].describe()

count     2.000000
mean     33.000000
std       2.828427
min      31.000000
25%      32.000000
50%      33.000000
75%      34.000000
max      35.000000
dtype: float64

In [38]:
s.loc[ ['Tue', 'Thu'] ].mean()

np.float64(32.75)

In [40]:
# option 1 for the first 3 days: use .iloc

s.iloc[:3].max()

np.int64(35)

In [41]:
s.head(3).max()

np.int64(35)

In [42]:
%timeit s.iloc[:3].max()

25.7 μs ± 312 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [43]:
%timeit s.head(3).max()

31.2 μs ± 1.13 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [44]:
s1 = Series([10, 20, 30, 40, 50], index=list('abcde'))
s2 = Series([100, 200, 300, 400, 500], index=list('edcba'))

s1 + s2  # it will add the values based on the index, not the position!

a    510
b    420
c    330
d    240
e    150
dtype: int64

In [45]:
s3 = Series([1000, 2000, 3000, 4000, 5000], index=list('abcab'))
s3

a    1000
b    2000
c    3000
a    4000
b    5000
dtype: int64

In [46]:
s1 + s3

a    1010.0
a    4010.0
b    2020.0
b    5020.0
c    3030.0
d       NaN
e       NaN
dtype: float64

In [47]:
s4 = Series([2, 4, 6, 8], index=list('abcd'))
s4

a    2
b    4
c    6
d    8
dtype: int64

In [48]:
s1 + s4

a    12.0
b    24.0
c    36.0
d    48.0
e     NaN
dtype: float64

s1 + s4 -- it gives us NaN (not a number) because there isn't a match on both sides

How can we get around this?

In Python, *every* operator is turned into a method call. So if you say `a + b`, this is really turned into `a.__add__(b)`.

This is useful to know, because a method can have additiona argu
